# Solution D1 · Reading the coefficients: odds ratios, collinearity and omitted variables

**Type**: Interpret

## Objective

Interpret the coefficients of a logistic correctly: tell odds from probability, measure collinearity (correlation and VIF), see
how the same coefficient changes **sign and size** depending on the other variables in the model, and verify it with a bootstrap interval.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
df["ratio"] = df["amount"] / df["income"]
df["unemployed"] = (df["contract_type"] == "unemployed").astype(int)
y = df["defaulted"].to_numpy()


def coefficients(columns, d=df, yy=y):
    """Coefficients (on standardized features) of an almost unregularized logistic."""
    m = make_pipeline(StandardScaler(), LogisticRegression(C=1e4, max_iter=2000)).fit(d[columns], yy)
    return pd.Series(m[-1].coef_[0], index=columns)


cols_a = ["ratio", "age", "job_tenure"]
print(f"{len(df)} customers, {y.mean():.1%} defaulters")

## Commented solution

In [ ]:
corr_age_tenure = df["age"].corr(df["job_tenure"])
vif = pd.Series({c: 1 / (1 - LinearRegression().fit(df[[a for a in cols_a if a != c]], df[c]).score(df[[a for a in cols_a if a != c]], df[c])) for c in cols_a})
print(f"age-tenure correlation: {corr_age_tenure:.3f}\nVIF:\n{vif.round(2).to_string()}")

In [ ]:
coef_a = coefficients(cols_a)
coef_b = coefficients(["ratio", "age"])
coef_c = coefficients(["ratio", "job_tenure"])
age_matters_alone = bool(abs(coef_b["age"]) > 0.1)
age_matters_given_tenure = bool(coef_a["age"] > 0.25)
print(pd.DataFrame({"A (age + tenure)": coef_a, "B (age only)": coef_b, "C (tenure only)": coef_c}).round(3))

Alone, age has no effect (coefficient 0.02), but **given tenure** it raises the risk (0.34). The same holds, with the opposite sign, for tenure. The pair behaves like
two variables working together: what matters are their **differences**, not the single values. See what happens with the bootstrap.

In [ ]:
rng = np.random.default_rng(0)
n = len(df)
boot_a, boot_b = [], []
for _ in range(200):
    i = rng.integers(0, n, n)
    boot_a.append(coefficients(cols_a, df.iloc[i], y[i])["age"])
    boot_b.append(coefficients(["ratio", "age"], df.iloc[i], y[i])["age"])
interval_a = tuple(np.percentile(boot_a, [2.5, 97.5]))
interval_b = tuple(np.percentile(boot_b, [2.5, 97.5]))
print("95% bootstrap interval of the coefficient of age: model A", np.round(interval_a, 3), "| model B", np.round(interval_b, 3))

In [ ]:
coef_e = coefficients(cols_a + ["unemployed"])
shrinkage = {c: abs(coef_e[c]) / abs(coef_a[c]) for c in ["age", "job_tenure"]}
print(pd.DataFrame({"A": coef_a, "A + unemployed": coef_e}).round(3))
print("ratio between the coefficients with and without `unemployed`:", {k: round(v, 2) for k, v in shrinkage.items()})

**What happened.** In the data whoever is unemployed has by construction `job_tenure = 0` and a much higher risk, and does not appear in model A. All else being equal in tenure, an
older customer is more often an unemployed one (so more at risk) and a low `job_tenure` serves to "detect" the unemployed. It is not an effect of age: it is an **omitted variable** that the two coefficients
split between them. When you add it, the two effects shrink to about a quarter of the previous value.

In [ ]:
model_a = make_pipeline(StandardScaler(), LogisticRegression(C=1e4, max_iter=2000)).fit(df[cols_a], y)
z0, w_ratio = model_a[-1].intercept_[0], model_a[-1].coef_[0][0]
base_prob = 1 / (1 + np.exp(-z0))
after_prob = 1 / (1 + np.exp(-(z0 + w_ratio)))
ratio_or = float(np.exp(w_ratio))
prob_delta = after_prob - base_prob
print(f"average customer: probability {base_prob:.3f} -> {after_prob:.3f} after +1 std dev of ratio (odds ratio {ratio_or:.2f})")
print(f"the odds get multiplied by {(after_prob / (1 - after_prob)) / (base_prob / (1 - base_prob)):.2f}, the probability by {after_prob / base_prob:.2f} (and goes up by {prob_delta:.3f}, not by {(ratio_or - 1) * base_prob:.3f})")

## Check

In [ ]:
with verify():
    check(0.8 < corr_age_tenure < 0.9, "age and job_tenure are highly correlated (about 0.85)", "np.corrcoef or df.corr()")
    check(vif is not None and vif["ratio"] < 1.5 and vif["age"] > 3 and vif["job_tenure"] > 3, "the VIF flags the collinearity between age and tenure (above 3), not for ratio", "VIF = 1 / (1 - R^2) from the linear regression of each feature on the other two")
    check(close(coef_a["age"], coefficients(cols_a)["age"]) and close(coef_b["age"], coefficients(["ratio", "age"])["age"]), "coefficients of the three models computed with the given function", "coefficients(columns)")
    check(age_matters_alone is False and age_matters_given_tenure is True, "alone age does not matter (coef close to 0); given tenure it does, and with a positive sign", "compare |coef_b['age']| with 0.1 and coef_a['age'] with 0.25")
    check(interval_a[0] > 0, "in model A the bootstrap interval of age lies entirely above zero", "2.5 and 97.5 percentiles of the 200 coefficients")
    check(interval_b[0] < 0 < interval_b[1], "in model B the interval of age contains zero", "age alone has no effect distinguishable from zero")
    check(shrinkage["age"] < 0.5 and shrinkage["job_tenure"] < 0.5, "by adding `unemployed`, both coefficients shrink to less than half", "|coef_e| / |coef_a| for age and job_tenure")
    check(close((after_prob / (1 - after_prob)) / (base_prob / (1 - base_prob)), ratio_or) and prob_delta < (ratio_or - 1) * base_prob,
          "the odds get multiplied by the odds ratio; the probability goes up far less than 'x times'", "prob = sigmoid(intercept) for the average customer; after_prob = sigmoid(intercept + coefficient of ratio)")
done()

## What you learned

- A coefficient is the effect **all else being equal among the variables in the model**. With `age` and `job_tenure` (correlation 0.85, VIF about 4) the coefficient of age goes from 0.02 (alone)
  to +0.34 (with tenure): same data, different "truths" depending on the model.
- The bootstrap tells you how **stable** the coefficients are on the sample, not whether they are **right**: in model A the interval of `age` is above zero (it is well estimated), and yet it comes from an omitted variable.
  Stable does not mean causal.
- With `unemployed` in the model the two coefficients collapse (to about a quarter of the previous value: 0.28 and 0.27). The rule: when two correlated coefficients have opposite signs and similar sizes,
  suspect a missing variable or a difference effect, and don't tell the story "age raises the risk".
- **Odds ratios** are multiplicative on the odds, not on the probabilities. For a customer with a 16% risk an odds ratio of 3.5 does not take the probability from 16% to 56% (3.5 times): it takes it to about 40%. With very small probabilities
  the two readings coincide (odds ≈ probability), with medium probabilities they don't.
- The VIF is an alarm, not a sentence: with VIF below 5 the estimates remain usable; the problem is interpretive, not numerical.

## Common pitfall

Reporting "the effect of age on default" by reading a coefficient of a model that also contains a highly correlated variable. A table of coefficients is a description
**of the model**, not of reality, and it is not causal even if the model is well estimated.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Two highly correlated variables have coefficients of opposite sign in your regression. How do you explain it to the team?</summary>

I say the coefficient measures the effect of one variable holding the other fixed, and when they are correlated the common part is split arbitrarily: their differences matter, not the single
values. I check the collinearity (VIF), try the model with only one at a time and with a constructed feature (a difference, a ratio), look for omitted variables that explain the opposite
sign, and if the goal is prediction I don't interpret the individual coefficients. For a causal effect you would need a different design, not a better model.

</details>